<a href="https://colab.research.google.com/github/Dani-Mol/InteligenciaArtificial/blob/main/Clase_08_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase Sep-08

## 1. Algoritmo A* (A-Star)

El algoritmo A* mejora a Dijkstra inyectando conocimiento del dominio mediante una **función heurística** $h(n)$. La prioridad en la cola ahora está dada por la ecuación:

$$f(n) = g(n) + h(n)$$

Donde $g(n)$ es el costo real acumulado y $h(n)$ es la distancia estimada al destino. Para que A* garantice devolver la ruta óptima (admisibilidad), la heurística **nunca debe sobreestimar** el costo real. En espacios euclidianos o cuadriculados, las distancias geométricas (Euclidiana o Manhattan) son heurísticas perfectas.

In [ ]:
import heapq
from collections import defaultdict
import math

def crear_grafo_vacio():
    return defaultdict(list)

grafo = crear_grafo_vacio()

# Grafo de ejemplo, donde cada tupla es (nodo_destino, peso_arista)
grafo['A'] = [('B', 1), ('C', 4)]
grafo['B'] = [('A', 1), ('C', 2), ('D', 5)]
grafo['C'] = [('A', 4), ('B', 2), ('D', 1)]
grafo['D'] = [('B', 5), ('C', 1)]

print("Estructura del grafo mediante defaultdict:")
for nodo, vecinos in grafo.items():
    print(f"Nodo {nodo}: {vecinos}")

Estructura del grafo mediante defaultdict:
Nodo A: [('B', 1), ('C', 4)]
Nodo B: [('A', 1), ('C', 2), ('D', 5)]
Nodo C: [('A', 4), ('B', 2), ('D', 1)]
Nodo D: [('B', 5), ('C', 1)]


In [ ]:
# Para A*, dotamos a los nodos de contexto espacial asignándoles coordenadas.
coordenadas = {
    'A': (0, 0),
    'B': (1, 0),
    'C': (0, 3),
    'D': (1, 4)
}

def heuristica_euclidiana(nodo1, nodo2):
    """Calcula la distancia en línea recta entre dos puntos cartesianos."""
    x1, y1 = coordenadas[nodo1]
    x2, y2 = coordenadas[nodo2]
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

def a_star(grafo, inicio, objetivo):
    # Nuevamente aprovechamos defaultdict para el costo real acumulado g(n)
    g_score = defaultdict(lambda: float('inf'))
    g_score[inicio] = 0

    padres = {}

    # Cola de prioridad: almacena (f_score, g_score, nodo_actual)
    # Es fundamental desempatar por el menor g_score si dos f_scores son iguales.
    f_inicial = heuristica_euclidiana(inicio, objetivo)
    pq = [(f_inicial, 0, inicio)]

    nodos_visitados = 0

    while pq:
        f_actual, g_actual, nodo_actual = heapq.heappop(pq)
        nodos_visitados += 1

        if nodo_actual == objetivo:
            break

        if g_actual > g_score[nodo_actual]:
            continue

        for vecino, peso in grafo[nodo_actual]:
            nuevo_g = g_actual + peso

            if nuevo_g < g_score[vecino]:
                g_score[vecino] = nuevo_g
                padres[vecino] = nodo_actual

                # El f(n) estimado para el vecino es su nuevo costo real + la heurística hacia el objetivo
                f_vecino = nuevo_g + heuristica_euclidiana(vecino, objetivo)
                heapq.heappush(pq, (f_vecino, nuevo_g, vecino))

    camino = []
    actual = objetivo
    while actual in padres:
        camino.append(actual)
        actual = padres[actual]
    camino.append(inicio)
    camino.reverse()

    return camino, g_score[objetivo], nodos_visitados

camino_a, costo_a, visitados_a = a_star(grafo, 'A', 'D')
print(f"A*       -> Ruta: {camino_a} | Costo total: {costo_a} | Nodos expandidos: {visitados_a}")

A*       -> Ruta: ['A', 'B', 'C', 'D'] | Costo total: 4 | Nodos expandidos: 4


## 2. Manejo de Python

Para este problema, lidiaremos con manipulación de matrices y recursividad. Es clave dominar los siguientes conceptos en Python:

*   **Inmutabilidad de los Strings**: En Python, los *strings* (cadenas de texto) no se pueden modificar una vez creados. No puedes hacer `texto[0] = 'A'`. Por lo tanto, convertiremos las filas del crucigrama en **listas de caracteres**, las cuales sí son mutables.
*   **List Comprehensions (Comprensión de listas)**: Usaremos una sintaxis compacta `[list(fila) for fila in cuadricula]` para transformar la cuadrícula original en una matriz 2D manipulable en una sola línea.
*   **Backtracking (Recursión)**: Es una técnica donde una función se llama a sí misma para explorar posibles soluciones. Si una ruta no funciona (una palabra no encaja más adelante), la función retrocede, deshace el último cambio y prueba otra opción.

In [ ]:
# Ejemplo introductorio de Mutabilidad y List Comprehension

texto_inmutable = "+---+"
# texto_inmutable[1] = 'A'  <-- Esto daría un TypeError en Python

# Solución: Convertir a lista de caracteres
lista_mutable = list(texto_inmutable)
lista_mutable[1] = 'A'
print("Lista mutable modificada:", lista_mutable)

# Reensamblando a string
texto_modificado = "".join(lista_mutable)
print("String final:", texto_modificado)

# List Comprehension para crear una matriz 2D rápidamente
cuadricula_inicial = ["+--+", "-++-"]
matriz_2d = [list(fila) for fila in cuadricula_inicial]
print("Matriz 2D lista para trabajar:", matriz_2d, "l")

# Funcion split(). convierte un string en una lista bajo un simbolo de sepracion
lista_palabras = "POLAND;LHASA;SPAIN;INDIA".split(';')
print(f"Lista de palabras: {lista_palabras}")

Lista mutable modificada: ['+', 'A', '-', '-', '+']
String final: +A--+
Matriz 2D lista para trabajar: [['+', '-', '-', '+'], ['-', '+', '+', '-']] l
Lista de palabras: ['POLAND', 'LHASA', 'SPAIN', 'INDIA']


## 2. Entendiendo el Problema

Tenemos una matriz exacta de $10 \times 10$. Las cruces (`+`) representan paredes impenetrables y los guiones (`-`) representan celdas vacías que deben ser llenadas con letras.

Además, nos dan una cadena de palabras separadas por punto y coma (ej. `POLAND;LHASA;SPAIN;INDIA`).

El desafío es ubicar todas las palabras en los espacios correspondientes. Dado que las palabras se cruzan, colocar una palabra condiciona a las demás. Si colocamos "POLAND" y luego descubrimos que "INDIA" no tiene espacio con las letras que se cruzaron, debemos **borrar** "POLAND" y probar en otro lugar. Esto es la esencia del *backtracking*.

## 3. Lógica y Estructura del Algoritmo

Dividiremos el problema en funciones auxiliares para mantener el código limpio:

1.  **`es_valido(r, c, palabra, direccion)`**: Verifica si una palabra cabe empezando en la fila `r` y columna `c`, ya sea horizontal o verticalmente. Comprueba que no se salga de la cuadrícula y que las letras coincidan con espacios vacíos (`-`) o con la misma letra si ya hay un cruce.
2.  **`colocar(r, c, palabra, direccion)`**: Escribe la palabra en la cuadrícula. Muy importante: debe recordar qué espacios estaban vacíos originalmente para poder borrarlos si nos equivocamos (sin borrar letras de palabras cruzadas válidas).
3.  **`backtrack(indice_palabra)`**: La función recursiva principal. Toma la palabra actual, busca toda la matriz de $10 \times 10$ intentando colocarla. Si la coloca, llama a `backtrack(indice + 1)`. Si esa llamada devuelve falso, **deshace** la colocación y sigue buscando.

In [ ]:
def crosswordPuzzle(crossword, words):
    # Convertimos los strings a una matriz 2D mutable
    grid = [list(row) for row in crossword]
    lista_palabras = words.split(';')


    # === FUNCIONES AUXILIARES DE VALIDACIÓN ===
    def es_valido_horizontal(r, c, palabra):
        if c + len(palabra) > 10:
            return False
        for i in range(len(palabra)):
            if grid[r][c+i] != '-' and grid[r][c+i] != palabra[i]:
                return False
        return True

    def es_valido_vertical(r, c, palabra):
        if r + len(palabra) > 10:
            return False
        for i in range(len(palabra)):
            if grid[r+i][c] != '-' and grid[r+i][c] != palabra[i]:
                return False
        return True

    # === FUNCIONES DE COLOCACIÓN (Retornan los índices modificados para el rollback) ===
    def colocar_horizontal(r, c, palabra):
        modificados = []
        for i in range(len(palabra)):
            if grid[r][c+i] == '-':
                grid[r][c+i] = palabra[i]
                modificados.append(i)
        return modificados

    def colocar_vertical(r, c, palabra):
        modificados = []
        for i in range(len(palabra)):
            if grid[r+i][c] == '-':
                grid[r+i][c] = palabra[i]
                modificados.append(i)
        return modificados

    # === FUNCIÓN PRINCIPAL DE BACKTRACKING ===
    def backtrack(indice=0):
        return 0

    return 0

# == PRUEBA LOCAL ==
crossword_input = [
    "+-++++++++",
    "+-++++++++",
    "+-++++++++",
    "+-----++++",
    "+-+++-++++",
    "+-+++-++++",
    "+++++-++++",
    "++------++",
    "+++++-++++",
    "+++++-++++"
]
words_input = "LONDON;DELHI;ICELAND;ANKARA"

resultado = crosswordPuzzle(crossword_input, words_input)
print("Crucigrama Resuelto:")
for fila in resultado:
    print(fila)

Crucigrama Resuelto:
+L++++++++
+O++++++++
+N++++++++
+DELHI++++
+O+++C++++
+N+++E++++
+++++L++++
++ANKARA++
+++++N++++
+++++D++++
